# BPE Tokenizer Visualization

Visualize the learned merges and token length distribution from a trained BPE tokenizer.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from bpe import BPETokenizer
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Train a tokenizer on the sample data
sample_path = Path.cwd().parent / "data" / "sample.txt"
text = sample_path.read_text(encoding="utf-8")

tok = BPETokenizer()
tok.train(text, vocab_size=1000)

print(f"Vocab size: {len(tok.vocab)}")
print(f"Merges learned: {len(tok.merges)}")
print(f"Compression ratio: {tok.compression_ratio(text):.2f} bytes/token")

In [ ]:
# Show the top 20 most frequent merges (by merge order)
top_merges = list(tok.merges.items())[:20]

fig, ax = plt.subplots(figsize=(10, 6))
labels = []
for (a, b), new_id in top_merges:
    a_str = tok.vocab[a].decode("utf-8", errors="replace")
    b_str = tok.vocab[b].decode("utf-8", errors="replace")
    merged = tok.vocab[new_id].decode("utf-8", errors="replace")
    labels.append(f"'{a_str}'+'{b_str}' -> '{merged}'")

y_pos = range(len(labels))
ax.barh(y_pos, [m[1] for m in top_merges], color='steelblue')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel('Merge order (lower = learned earlier = more frequent)')
ax.set_title('Top 20 Most Frequent Merges')
plt.tight_layout()
plt.show()

In [ ]:
# Token length distribution
lengths = [len(tok.vocab[i]) for i in range(256, len(tok.vocab))]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(lengths, bins=range(1, max(lengths) + 2), color='coral', edgecolor='black')
ax.set_xlabel('Token length (bytes)')
ax.set_ylabel('Count')
ax.set_title('Token Length Distribution (merged tokens only)')
plt.tight_layout()
plt.show()

In [ ]:
# Encode a sample and show the tokens
sample = "The quick brown fox jumps over the lazy dog."
ids = tok.encode(sample)
tokens = [tok.token_str(i) for i in ids]

print(f"Text: {sample}")
print(f"Tokens ({len(ids)}): {tokens}")
print(f"Decoded: {tok.decode(ids)}")
print(f"Match: {tok.decode(ids) == sample}")